# SPBE - Retrieval Augmented Generation with Knowledge Graph

# Import Dependency

In [ ]:
# endpoint=https://spbeopenai.openai.azure.com/

In [17]:
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="", temperature=0)
# structured_llm = llm.with_structured_output(method="json_mode", include_raw=True)

# structured_llm.invoke("")

In [ ]:
from langchain_core.runnables import  RunnablePassthrough, RunnableParallel # type: ignore
from langchain_core.prompts import ChatPromptTemplate # type: ignore
from pydantic import BaseModel, Field # type: ignore
from langchain_core.output_parsers import StrOutputParser # type: ignore
from langchain.text_splitter import RecursiveCharacterTextSplitter # type: ignore
from langchain_groq import ChatGroq # type: ignore
from langchain_experimental.graph_transformers import LLMGraphTransformer # type: ignore
from neo4j import GraphDatabase # type: ignore
from yfiles_jupyter_graphs import GraphWidget # type: ignore
from langchain_neo4j import Neo4jGraph, Neo4jVector # type: ignore
from langchain_community.document_loaders import UnstructuredMarkdownLoader # type: ignore
from langchain_text_splitters import MarkdownHeaderTextSplitter # type: ignore
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars # type: ignore
from langchain_ollama import OllamaEmbeddings # type: ignore
from langchain_openai import AzureChatOpenAI # type: ignore

import os
from neo4j import Driver # type: ignore

from dotenv import load_dotenv # type: ignore
import pandas as pd  # type: ignore

load_dotenv()

True

# Connection

In [2]:
# Integration Azure Open AI in Langchain
llm_openai_azure = AzureChatOpenAI(
    openai_api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
    openai_api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)

# Uji coba
response = llm_openai_azure.invoke("Apa itu LangChain? jawab dengan singkat saja")
print(response.content)

LangChain adalah framework open-source untuk membangun aplikasi berbasis Large Language Models, dengan dukungan prompt chaining, memory, integrasi data eksternal, dan agent tools.


In [3]:
# Initialize Neo4J connection

NEO4J_URI = os.environ.get('NEO4J_URI_regguy')
NEO4J_USERNAME = os.environ.get('NEO4J_USERNAME_regguy')
NEO4J_PASSWORD = os.environ.get('NEO4J_PASSWORD_regguy')
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE_regguy') 

graph = Neo4jGraph(
            url=NEO4J_URI,
            username=NEO4J_USERNAME,
            password=NEO4J_PASSWORD,
            database=NEO4J_DATABASE,
            refresh_schema=False
            )

print("connected to NeoJ Instance") 

connected to NeoJ Instance


# Load and Process Knowledge Graph (300 Minutes estimate)

In [5]:
# Load the SPBE Document

markdown_path = "data/parsed_5. Pedoman Menteri PANRB NO 3 Tahun 2024 Pedoman Tata Cara Pemantauan dan Evaluasi SPBE.md" 
loader = UnstructuredMarkdownLoader(markdown_path, mode="elements")

spbe_md = loader.load()
print(f"Number of documents: {len(spbe_md)}\n")

for document in spbe_md[:3]:
    print(f"{document.page_content}\n")


Number of documents: 1826

SALINAN

MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA

PEDOMAN MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA NOMOR 3 TAHUN 2024 TENTANG TATA CARA PEMANTAUAN DAN EVALUASI SISTEM PEMERINTAHAN BERBASIS ELEKTRONIK



In [6]:
# menggabungkan semua teks dari `spbe_md` menjadi satu string
markdown_text = "\n".join(doc.page_content for doc in spbe_md)

headers_to_split_on = [
    ("# SALINAN", "Title 1"),
    ("# BAB", "BAB"),
    ("## A", "SubBAB A"),
    ("## B", "SubBAB B"),
    ("## C", "SubBAB C"),
    ("## D", "SubBAB D"),
    ("## E", "SubBAB E"),
    ("## F", "SubBAB F"),
    ("## G", "SubBAB G"),
    ("# Indikator", "Indikator"),
    ("## Level", "Level"), 
]

# MD splits
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False, return_each_line=True
)
md_header_splits = markdown_splitter.split_text(markdown_text)

# Char-level splits
chunk_size = 1500
chunk_overlap = 100
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

# Split
documents = text_splitter.split_documents(md_header_splits)
len(documents)

425

In [6]:
type(documents[0])

langchain_core.documents.base.Document

In [7]:
for i, chunk in enumerate(documents):
    print(f"Chunk: {i+1}: \n{chunk.page_content}\n")

Chunk: 1: 
SALINAN
MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA
PEDOMAN MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA NOMOR 3 TAHUN 2024 TENTANG TATA CARA PEMANTAUAN DAN EVALUASI SISTEM PEMERINTAHAN BERBASIS ELEKTRONIK
DENGAN RAHMAT TUHAN YANG MAHA ESA
MENTERI PENDAYAGUNAAN APARATUR NEGARA DAN REFORMASI BIROKRASI REPUBLIK INDONESIA,
BAB I PENDAHULUAN
A. Latar Belakang
Sistem Pemerintahan Berbasis Elektronik (SPBE) merupakan penyelenggaraan pemerintahan yang memanfaatkan teknologi informasi dan komunikasi dalam rangka meningkatkan kualitas layanan administrasi pemerintahan dan pelayanan publik yang efisien dan optimal, merupakan amanat pelaksanaan Peraturan Presiden Republik Indonesia Nomor 95 Tahun 2018 tentang Sistem Pemerintahan Berbasis Elektronik (Perpres SPBE). Pelaksanaan SPBE menjadi fondasi serta sebagai pengungkit (enabler) dari reformasi birokrasi melalui pelaksanaan transformasi digital dan Satu Data Indo

In [5]:
# Initialize Model
# llm_raw = ChatGroq(groq_api_key=os.environ.get("QROQ_API_KEY_SPBE_GRAPH_MYUNESA"),
#                     model_name="llama3-70b-8192", 
#                     temperature=0)

# structured_llm = llm_raw.with_structured_output(method="json_mode", include_raw=True)

In [8]:
# Estimasi token dan batching + pengiriman ke Neo4j secara efisien dan aman
import time
from tqdm import tqdm # type: ignore
from langchain_core.messages.utils import count_tokens_approximately # type: ignore

# Estimasi token per dokumen
def estimate_tokens(text):
    return count_tokens_approximately(text)

# Print token estimation for each document and total
total_tokens = 0
for i, doc in enumerate(documents):
    tokens = estimate_tokens(doc.page_content)
    total_tokens += tokens

print(f"Total tokens across all documents: {total_tokens}")

# Siapkan parameter batching
MAX_TOKEN_PER_BATCH = 3000
SLEEP_TIME = 25  # detik antar batch (aman dari rate limit)

# Membagi dokumen menjadi batch berdasarkan token
batches = []
current_batch = []
current_token_count = 0

for doc in documents:
    token_count = estimate_tokens(doc.page_content)

    # Jika melebihi batas, push batch dan reset
    if current_token_count + token_count > MAX_TOKEN_PER_BATCH:
        batches.append(current_batch)
        current_batch = [doc]
        current_token_count = token_count
    else:
        current_batch.append(doc)
        current_token_count += token_count

# Tambahkan batch terakhir jika masih ada
if current_batch:
    batches.append(current_batch)

print(f"Total batch yang akan diproses: {len(batches)}")

Total tokens across all documents: 2143700
Total batch yang akan diproses: 395


In [9]:
# Proses setiap batch

# Define node and relationship
spbe_nodes = [
    "SPBE",
    "Indeks",
    "Domain",
    "Aspek",
    "Indikator",
    "Kuesioner",
    "Deskripsi_Indikator",
    "Ketentuan_Penilaian",
    "Contoh_Bukti_Dukung",
    "Level",
    "Kriteria_Level",
    "Kriteria_Pemenuhan_Level",
    "Kriteria_Bukti_Dukung",
    "Contoh_Kaidah",
    "Bab_Pedoman",
    "Poin_Pedoman",
    "Kriteria_Kaidah"
]

spbe_relationships = [
    ("SPBE", "Memiliki_Domain", "Domain"),
    ("Domain", "Memiliki_Aspek", "Aspek"),
    ("Aspek", "Memiliki_Indikator", "Indikator"),
    ("Indikator", "Memiliki_Level", "Level"),
    ("Level", "Memiliki_Kriteria_Level", "Kriteria_Level"),
    ("Level", "Memiliki_Kriteria_Pemenuhan", "Kriteria_Pemenuhan_Level"),
    ("Level", "Memiliki_Kriteria_Bukti_Dukung", "Kriteria_Bukti_Dukung"),
    ("Indikator", "Memiliki_Kuesioner", "Kuesioner"),
    ("Indikator", "Memiliki_Deskripsi", "Deskripsi_Indikator"),
    ("Indikator", "Memiliki_Ketentuan_Penilaian", "Ketentuan_Penilaian"),
    ("Indikator", "Memiliki_Contoh_Bukti_Dukung", "Contoh_Bukti_Dukung"),
    ("SPBE", "Memiliki_Bab", "Bab_Pedoman"),
    ("Bab_Pedoman", "Memiliki_SubBab", "Poin_Pedoman"),
    ("Poin_Pedoman", "Memiliki_SubPoin", "Poin_Pedoman"),
    ("SPBE", "Memiliki_Indeks", "Indeks"),
    ("Contoh_Kaidah", "Contoh_Kaidah_Memiliki_Kriteria", "Kriteria_Kaidah"),
    ("Bab_Pedoman", "Memiliki_SubBab", "Poin_Pedoman"),
    ("Poin_Pedoman", "Memiliki_SubPoin", "Poin_Pedoman"),
]

graph_transformer = LLMGraphTransformer(
      llm=llm_openai_azure,
      allowed_nodes=spbe_nodes,
      allowed_relationships=spbe_relationships,
      strict_mode=False,
      node_properties=True,
      relationship_properties=True
      )

# Menyimpan semua hasil untuk referensi
graph_documents = []
MAX_RETRIES = 3

for i, batch in enumerate(tqdm(batches, desc="Processing Batches")):

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = await graph_transformer.aconvert_to_graph_documents(batch)
            graph_documents.extend(result)

            # Tambahkan langsung ke Neo4j
            graph.add_graph_documents(result, baseEntityLabel=True, include_source=True)
            break  # keluar dari retry loop jika berhasil

        except Exception as e:
            print(f"[Batch {i+1}] Percobaan {attempt} gagal: {e}")
            if attempt == MAX_RETRIES:
                print(f"❌ Batch {i+1} gagal setelah {MAX_RETRIES} percobaan. Lewati.")
            else:
                time.sleep(SLEEP_TIME)

    time.sleep(SLEEP_TIME)

print("\n✅ Selesai memproses semua batch dan menambahkan ke Neo4j.")

Processing Batches: 100%|██████████| 395/395 [5:03:24<00:00, 46.09s/it]  


✅ Selesai memproses semua batch dan menambahkan ke Neo4j.


In [10]:
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Sistem Pemerintahan Berbasis Elektronik (Spbe)', type='Spbe', properties={'deskripsi': 'penyelenggaraan pemerintahan yang memanfaatkan teknologi informasi dan komunikasi dalam rangka meningkatkan kualitas layanan administrasi pemerintahan dan pelayanan publik yang efisien dan optimal', 'amanat': 'Peraturan Presiden Republik Indonesia Nomor 95 Tahun 2018 tentang Sistem Pemerintahan Berbasis Elektronik', 'tujuan': 'menciptakan birokrasi pemerintah yang integratif, dinamis, transparan, dan inovatif pada Instansi Pusat dan Pemerintah Daerah'}), Node(id='Bab I Pendahuluan', type='Bab_pedoman', properties={'judul': 'Pendahuluan'}), Node(id='A. Latar Belakang', type='Poin_pedoman', properties={'judul': 'Latar Belakang'})]
Relationships:[Relationship(source=Node(id='Sistem Pemerintahan Berbasis Elektronik (Spbe)', type='Spbe', properties={}), target=Node(id='Bab I Pendahuluan', type='Bab_pedoman', properties={}), type='MEMILIKI_BAB', properties={}), Relationship(source=Node(id=

# Vector Index and Embedding

In [4]:
# Initialize Embedding
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)
# Create Neo4J Vector Index
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    url = os.environ.get('NEO4J_URI_regguy'),
    username = os.environ.get('NEO4J_USERNAME_regguy'),
    password = os.environ.get('NEO4J_PASSWORD_regguy'),
)

In [5]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI_regguy"],
        auth = (os.environ["NEO4J_USERNAME_regguy"],
                os.environ["NEO4J_PASSWORD_regguy"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id`
    IF NOT EXISTS
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session() as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    pass

# Close the driver connection
driver.close()

Fulltext index created successfully.


In [8]:
from typing import Optional

class Entities(BaseModel):
    """Identifying information about entities."""

    names: Optional[list[str]] = Field(default_factory=list, description="Nama orang, organisasi, atau entitas bisnis")
    kata_kunci_spbe: Optional[list[str]] = Field(default_factory=list, description="Istilah atau frasa penting yang berkaitan dengan SPBE, misalnya arsitektur SPBE, layanan digital, transformasi pemerintahan digital, peraturan, domain, aspek, indikator, level, kematangan, kriteria dll, tahun indeks, untuk kata kunci tahun, ekstrak format seperti '2021', '2022', '2023'. ")

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Kamu adalah asisten yang mengekstrak entitas dari dokumen kebijakan SPBE (Sistem Pemerintahan Berbasis Elektronik). "
        "Ekstrak dan kategorikan semua nama organisasi, domain SPBE, aspek SPBE, indikator evaluasi, dan istilah atau kata kunci penting dari teks berikut."
    ),
    (
        "human",
        "Gunakan format berikut untuk mengekstrak informasi dari input:\n{question}"
    )
])


entity_chain = prompt | llm_openai_azure.with_structured_output(Entities)

In [9]:
entity_chain.invoke("Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan")

Entities(names=['IPPD', 'Instansi Pusat', 'Pemerintah Daerah'], kata_kunci_spbe=['Manajemen Risiko SPBE', 'Pedoman Manajemen Risiko SPBE', 'Sistem Pemerintahan Berbasis Elektronik'])

In [10]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text
    search. It processes the input string by splitting it into words and 
    appending a similarity threshold (~2 changed characters) to each
    word, then combines them using the AND operator. Useful for mapping
    entities from user questions to database values, and allows for some 
    misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()


# Fulltext index query
def graph_retriever(question: str, max_entities: int = 5) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})

    all_entities = []
    for field in ["names", "kata_kunci_spbe"]:
        values = getattr(entities, field, [])
        if values:
            all_entities.extend(values)

    selected_entities = all_entities[:max_entities]

    for entity in selected_entities:
        response = graph.query(
            """
            CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit:2})
            YIELD node
            OPTIONAL MATCH (node)-[r]->(neighbor)
            RETURN
              node.id AS node_id,
              properties(node) AS props,
              labels(node) AS labels,
              COLLECT([type(r), neighbor.id]) AS relationships
            LIMIT 20
            """,
            {"query": generate_full_text_query(entity)},
        )
        for el in response:
            result += f"🔹 Node: {el['node_id']}\n"
            result += f"   Labels: {', '.join(el['labels'])}\n"

            for key, value in el['props'].items():
                result += f"     {key}: {value}\n"

            relationships = el.get("relationships", [])
            if relationships:
                result += f"   Relationships:\n"
                for rel in relationships:
                    if rel and len(rel) == 2:
                        rel_type, neighbor = rel
                        result += f"     ➡️ -[{rel_type}]-> {neighbor}\n"

            result += "\n"

    return result or "[INFO] Tidak ada hasil graph yang ditemukan."


In [11]:
print(graph_retriever("Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan"))

🔹 Node: Jumlah Responden (Ippd)
   Labels: __Entity__, Indeks
     id: Jumlah Responden (Ippd)
     2021: 517
     2023: 621
     2022: 554
   Relationships:
     ➡️ -[None]-> None

🔹 Node: Jumlah Ippd Kategori “Baik”
   Labels: __Entity__, Indeks
     id: Jumlah Ippd Kategori “Baik”
     2021: 159
     2023: 388
     2022: 237
   Relationships:
     ➡️ -[None]-> None

🔹 Node: Instansi Pusat/Pemerintah Daerah
   Labels: __Entity__, Level
     id: Instansi Pusat/Pemerintah Daerah
   Relationships:
     ➡️ -[MEMILIKI_KRITERIA_PEMENUHAN]-> Kriteria Pemenuhan Level Instansi Pusat/Pemerintah Daerah
     ➡️ -[MEMILIKI_KRITERIA_PEMENUHAN]-> Telah Memiliki Dokumen Arsitektur Spbe Sesuai Standar Dan Selaras Dengan Arsitektur Spbe Nasional Yang Sudah Ditetapkan Melalui Keputusan Pimpinan Instansi Pusat/Kepala Daerah Dan Sudah Memiliki Konten Metadata Arsitektur Spbe Pada Sistem Informasi
     ➡️ -[MEMILIKI_KRITERIA_PEMENUHAN]-> Melakukan Perbaikan Penerapan Manajemen Risiko Spbe Dan/Atau Telah T

In [12]:
def full_retriever(question: str):
    graph_data = graph_retriever(question)
    vector_data = [el.page_content for el in vector_index.similarity_search(question, k=2)]
    final_data = f"""Graph data:
{graph_data}
vector data:
{"#Document ". join(vector_data)}
    """
    return final_data

In [13]:
print(full_retriever("Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan"))

Graph data:
🔹 Node: Jumlah Responden (Ippd)
   Labels: __Entity__, Indeks
     id: Jumlah Responden (Ippd)
     2021: 517
     2023: 621
     2022: 554
   Relationships:
     ➡️ -[None]-> None

🔹 Node: Jumlah Ippd Kategori “Baik”
   Labels: __Entity__, Indeks
     id: Jumlah Ippd Kategori “Baik”
     2021: 159
     2023: 388
     2022: 237
   Relationships:
     ➡️ -[None]-> None

🔹 Node: Instansi Pusat/Pemerintah Daerah
   Labels: __Entity__, Level
     id: Instansi Pusat/Pemerintah Daerah
   Relationships:
     ➡️ -[MEMILIKI_KRITERIA_PEMENUHAN]-> Kriteria Pemenuhan Level Instansi Pusat/Pemerintah Daerah
     ➡️ -[MEMILIKI_KRITERIA_PEMENUHAN]-> Telah Memiliki Dokumen Arsitektur Spbe Sesuai Standar Dan Selaras Dengan Arsitektur Spbe Nasional Yang Sudah Ditetapkan Melalui Keputusan Pimpinan Instansi Pusat/Kepala Daerah Dan Sudah Memiliki Konten Metadata Arsitektur Spbe Pada Sistem Informasi
     ➡️ -[MEMILIKI_KRITERIA_PEMENUHAN]-> Melakukan Perbaikan Penerapan Manajemen Risiko Spbe Dan/

In [14]:
SPBE_prompt_template = """
Anda adalah asisten asesor SPBE dinamai SPBEBOT, khususnya dalam membantu menjawab pertanyaan terkait SPBE dan juga auditing Sistem Pemerintahan Berbasis Elektronik (SPBE) Indonesia.

terkait Sistem Pemerintahan Berbasis Elektronik (SPBE). Didalamnya ada 4 Domain yang disingkat D (contoh: D1,D2,D3,D4), lalu ada 8 Aspek yang disingkat A (contoh: A2, A5) dan 47 Indikator yang disingkat ID (contoh: ID-3, ID-23). jadi struktur dari SPBE ada 3 unsur tadi domain, aspek, dan indikator, dan penilaian indikator ada kriteria level nya yaitu level 1-5,  dengan level 5 paling tinggi nya.
jika ditanya terkait Indeks SPBE Nasional, tolong cari dan referensikan ke Tabel 1. Indeks SPBE Nasional (2021 - 2023)

Ada 4 Domain utama SPBE:
1. Domain 1: Kebijakan
2. Domain 2: Tata Kelola
3. Domain 3: Manajemen
4. Domain 4: Layanan

Ada 8 Aspek utama SPBE:
1. Aspek 1: Kebijakan Tata Kelola SPBE
2. Aspek 2: Perencanaan Strategis SPBE
3. Aspek 3: Teknologi Informasi dan Komunikasi
4. Aspek 4: Penyelenggaraan SPBE
5. Aspek 5: Penerapan Manajemen SPBE
6. Aspek 6: Audit TIK
7. Aspek 7: Layanan Administrasi Pemerintahan
8. Aspek 8: Layanan Publik

Ada 47 Indikator utama SPBE:
1. Indikator 1-10 berada di Domain Kebijakan (bobot 13%)
2. Indikator 11-20 berada di Domain Tata Kelola (bobot 25%)
3. Indikator 21-31 berada di Domain Manajemen (bobot 16,5%)
4. Indikator 32-47 berada di Domain Layanan (bobot 45,5%)

Ada 5 Tingkat Kematangan Domain Kebijakan, Tata Kelola, dan Manajemen:
1. Rintisan
2. Terkelola
3. Terdefinisi
4. Terpadu dan Terukur
5. Optimum

Sedangkan untuk Domain Layanan:
1. Informasi
2. Interaksi
3. Transaksi
4. Kolaborasi
5. Optimum

## 🎯 Aturan Menjawab dan Tugas:
1. Dalam memberikan penilaian level berapa sebuah indikator, maka pastikan memenuhi kriteria pemenuhan level dan kriteria bukti dukung, ini harus ada jika melakukan audit, kalau tidak ada maka mintakan kepada pengguna hal tersebut.
2. Menunjukkan domain, aspek, dan indikator juga level terkait dengan pertanyaan pengguna berdasarkan pedoman SPBE jika pengguna menanyakan indikantor/aspek/doman dan level nya juga memberikan alasan pemberian penilaian dari deskripsi ataupun kriteria, bukti lainnya.
3. Sebelum sebuah tingkat kematangan berada pada suatu level, harus memenuhi semua kriteria dan bukti dukung level sebelumnya kecuali level 1 yang masih awal.
4. Kamu akan diberikan konteks dibawah, jadi pastikan hal yang dibutuhkan untuk penilaian tersedia, jangan menjawab jika konteks tidak tersedia
5. jawab dengan konkrit tidak terlalu panjang dan jelas.

Ingat jika pertanyaan menentukan level penilaian dari indeks SPBE maka format jawaban nya bisa seperti contoh format jawaban dibawah ini:

Berdasarkan pedoman pemantauan dan evaluasi SPBE.....dengan diberikannya informasi tahapan, kriteria pemenuhan level dan bukti dukung maka indeks SPBE diberikan: 

{{ 
  "Penilaian": {{
    "Level": 5,
    "Indikator": 3,
    "Aspek": 2,
    "Domain": 1
  }}
}}

alasan penilaian diatas karena.....level.....indikator....aspek....domain...sesuai standard dari pedoman evaluasi SPBE.
yang titik diatas kamu harus isi alasannya, dan sesuai format

kalau terkait pertanyaan kebijakan, peraturan, kriteria, nilai indeks tahun, nama domain, cukup jawab secara konkrit saja tanpa perlu format jawaban diatas

## 🔍 Konteks yang diberikan dari dokumen:
{context}

## ❓ Pertanyaan pengguna:
{question}

## ✅ Jawaban SPBEBOT:
"""

In [15]:
# template = """Answer the question based only on the following context:
# {context}

# Question: {question}
# Use natural language and be concise.
# Answer:"""
prompt = ChatPromptTemplate.from_template(SPBE_prompt_template)

llm = ChatGroq(
    groq_api_key=os.environ.get("QROQ_API_KEY_SPBE_GRAPH_MYUNESA"),
    model_name="llama-3.3-70b-versatile", 
    temperature=0
    )

graphrag_chain = (
    RunnableParallel(
        {
            "context": full_retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [16]:
graphrag_chain.invoke("Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan")

'Permenpanrb No 5 Tahun 2020 Tentang Pedoman Manajemen Risiko SPBE.'

# Evaluasi GraphRAG with RAGAS

In [ ]:
import pickle
import time
import pandas as pd # type: ignore
from langchain_groq import ChatGroq # type: ignore
from ragas import evaluate, EvaluationDataset # type: ignore
from ragas.run_config import RunConfig # type: ignore
from ragas.llms import LangchainLLMWrapper # type: ignore
from ragas.embeddings import LangchainEmbeddingsWrapper # type: ignore
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings # type: ignore
from ragas.metrics import ( # type: ignore
    faithfulness, 
    answer_relevancy, 
    context_recall, 
    context_precision
    )

In [18]:
def prepare_ragas_data_graphrag(question, reference):
    """
    Menyiapkan data untuk evaluasi RAGAS dari GraphRAG chain
    """
    try:
        # Get context from full_retriever
        context = full_retriever(question)
        
        # Get response from GraphRAG chain
        response = graphrag_chain.invoke(question)
        
        return {
            "user_input": question,
            "retrieved_contexts": [context],  # RAGAS expects list of contexts
            "response": response,
            "reference": reference
        }
    except Exception as e:
        print(f"Error in prepare_ragas_data_graphrag: {str(e)}")
        return None

In [19]:
def evaluate_graphrag_with_ragas(questions, references):
    """
    Evaluasi GraphRAG menggunakan RAGAS
    """
    ragas_dataset = []
    
    for i, question in enumerate(questions):
        print(f"Processing question {i+1}/{len(questions)}: {question[:50]}...")
        
        # Get reference if available
        reference = references[i] if references and i < len(references) else None
        
        # Prepare RAGAS data
        ragas_data = prepare_ragas_data_graphrag(question, reference)
        
        if ragas_data:
            ragas_dataset.append(ragas_data)
            print(f"✅ Question {i+1} processed successfully")
        else:
            print(f"❌ Failed to prepare RAGAS data for question {i+1}")

        # Jeda untuk menghindari rate limiting
        print("⏳ Menunggu 10 detik...")
        time.sleep(10)
    
    return ragas_dataset

In [20]:
# Test questions untuk evaluasi GraphRAG
questions_for_evaluation = [
    """Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan""",
    """Jika "Rencana dan Anggaran SPBE Instansi Pusat Pemerintah Daerah telah terpadu dan dapat dikendalikan oleh unit kerja/perangkat daerah yang menjalankan fungsi perencanaan dan penganggaran dan telah direviu serta dievaluasi secara periodik:" Maka dapat diberikan level...""",
    "Berapa nilai indeks domain yang masih dibawah target pada tahun 2021-2023?",
    """Sebuah IPPD melampirkan Peta Rencana SPBE yang telah didokumentasikan secara formal, dan mengklaim memiliki dokumen Peta Rencana SPBE yang telah mengatur seluruh muatan Peta Rencana SPBE Instansi Pusat Pemerintah Daerah. Dokumen Peta Rencana yang diunggah berisikan muatan Peta Rencana secara lengkap antara lain Tata Kelola SPBE, Manajemen SPBE, Layanan SPBE, Arsitektur SPBE, Aplikasi SPBE, Keamanan SPBE dan Audit TIK. Maka level yang pantas diberikan adalah...
""",
    """Dibawah ini merupakan domain-domain dari Arsitektur SPBE berdasarkan Perpres SPBE, kecuali (pilih salah satu):

- domain arsitektur Proses Bisnis
- domain arsitektur Manajemen SPBE 
- domain arsitektur Infrastruktur SPBE
- domain arsitektur Aplikasi SPBE
- domain arsitektur Keamanan SPBE
- domain arsitektur Layanan SPBE
"""
]

In [21]:
references = [
    """PermenPANRB No. 5 Tahun 2020 memberikan pedoman umum bagi Instansi Pusat dan Pemerintah Daerah dalam melaksanakan SPBE, termasuk penerapan Manajemen Risiko SPBE
    """,
    """Berdasarkan pedoman pemantauan dan evaluasi SPBE, dengan diberikannya informasi kriteria-kriteria, maka indeks SPBE yang diberikan: 
 
Penilaian: 
Level: 4,
Indikator: 13,
Aspek: 2,
Domain: 2

Alasan penilaian diatas karena Rencana dan Anggaran SPBE Instansi Pusat/Pemerintah Daerah telah terpadu dan dapat dikendalikan oleh unit kerja/perangkat daerah yang menjalankan fungsi perencanaan dan penganggaran dan telah direviu serta dievaluasi secara periodik, sehingga sesuai dengan kriteria Level 4 pada Domain Tata Kelola, Aspek Perencanaan Strategis SPBE, dan Indikator 13 tentang tingkat kematangan keterpaduan rencana dan anggaran SPBE.
""",
    """Nilai indeks domain yang masih di bawah target (<2,60) pada tahun 2021-2023 adalah:

1. Indeks Domain Tata Kelola: 
   - Tahun 2021: 1,89
   - Tahun 2022: 1,85
   - Tahun 2023: 2,29

2. Indeks Domain Manajemen: 
   - Tahun 2021: 1,23
   - Tahun 2022: 1,32
   - Tahun 2023: 1,66
   """,
    """Berdasarkan pedoman pemantauan dan evaluasi SPBE, dengan diberikannya informasi kriteria-kriteria, maka indeks SPBE yang diberikan: 
 
Penilaian: 
Level: 3,
Indikator: 12,
Aspek: 2,
Domain: 2

Alasan penilaian diatas karena dokumen Peta Rencana SPBE telah mengatur seluruh muatan Peta Rencana SPBE Instansi Pusat/Pemerintah Daerah secara lengkap (Tata Kelola SPBE, Manajemen SPBE, Layanan SPBE, Infrastruktur SPBE, Aplikasi SPBE, Keamanan SPBE, Audit Teknologi SPBE dan Audit TIK) dan dokumen Peta Rencana SPBE telah didokumentasikan secara formal, sehingga sesuai dengan kriteria Level 3 pada Domain Tata Kelola, Aspek Perencanaan Strategis SPBE, dan Indikator 12 tentang Tingkat Kematangan Peta Rencana SPBE Instansi Pusat/Pemerintah Daerah. Namun, perlu diperhatikan bahwa untuk mencapai Level 4, IPPD harus memenuhi kriteria tambahan, yaitu dokumen Peta Rencana SPBE telah diterapkan secara konsisten melalui rencana kerja dan anggaran 3 (tiga) tahun terakhir, dan dokumen Peta Rencana SPBE telah dilakukan reviu dan evaluasi secara periodik.
""",
    "Domain Arsitektur Manajemen SPBE tidak termasuk dalam daftar domain arsitektur SPBE yang ditetapkan dalam Perpres SPBE. Domain Manajemen SPBE sebenarnya merupakan salah satu aspek dalam Sistem Pemerintahan Berbasis Elektronik (SPBE), bukan domain arsitektur SPBE."
]

In [22]:
"""
Menjalankan evaluasi GraphRAG dengan RAGAS
"""
print("🚀 Memulai evaluasi GraphRAG dengan RAGAS...")

# Menjalankan evaluasi
ragas_dataset = evaluate_graphrag_with_ragas(questions_for_evaluation, references)

# Menampilkan hasil
print(f"\n📊 Dataset RAGAS berhasil disiapkan dengan {len(ragas_dataset)} sampel")
for i, data in enumerate(ragas_dataset):
    print(f"\n--- Sampel {i+1} ---")
    print(f"Question: {data['user_input']}")
    print(f"Context Length: {len(str(data['retrieved_contexts']))} characters")
    print(f"Response: {data['response'][:100]}...")
    if data['reference']:
        print(f"Reference: {data['reference'][:100]}...")

🚀 Memulai evaluasi GraphRAG dengan RAGAS...
Processing question 1/5: Dalam penerapan Manajemen Risiko SPBE, masing-masi...
✅ Question 1 processed successfully
⏳ Menunggu 10 detik...
Processing question 2/5: Jika "Rencana dan Anggaran SPBE Instansi Pusat Pem...
✅ Question 2 processed successfully
⏳ Menunggu 10 detik...
Processing question 3/5: Berapa nilai indeks domain yang masih dibawah targ...
✅ Question 3 processed successfully
⏳ Menunggu 10 detik...
Processing question 4/5: Sebuah IPPD melampirkan Peta Rencana SPBE yang tel...
✅ Question 4 processed successfully
⏳ Menunggu 10 detik...
Processing question 5/5: Dibawah ini merupakan domain-domain dari Arsitektu...
✅ Question 5 processed successfully
⏳ Menunggu 10 detik...

📊 Dataset RAGAS berhasil disiapkan dengan 5 sampel

--- Sampel 1 ---
Question: Dalam penerapan Manajemen Risiko SPBE, masing-masing IPPD (Instansi Pusat dan Pemerintah Daerah) dapat mengacu pada Pedoman Manajemen Risiko SPBE yang ditetapkan dalam kebijakan
Context 

In [23]:
# Konversi ke DataFrame
df_ragas = pd.DataFrame(ragas_dataset)
df_ragas

,user_input,retrieved_contexts,response,reference
0,"Dalam penerapan Manajemen Risiko SPBE, masing-...",[Graph data:\n🔹 Node: Jumlah Responden (Ippd)\...,Permenpanrb No 5 Tahun 2020 Tentang Pedoman Ma...,PermenPANRB No. 5 Tahun 2020 memberikan pedoma...
1,"Jika ""Rencana dan Anggaran SPBE Instansi Pusat...",[Graph data:\n🔹 Node: Pelayanan Administrasi P...,Berdasarkan pedoman pemantauan dan evaluasi SP...,Berdasarkan pedoman pemantauan dan evaluasi SP...
2,Berapa nilai indeks domain yang masih dibawah ...,[Graph data:\n🔹 Node: Indeks Domain Manajemen\...,Nilai indeks domain yang masih dibawah target ...,Nilai indeks domain yang masih di bawah target...
3,Sebuah IPPD melampirkan Peta Rencana SPBE yang...,[Graph data:\n🔹 Node: Jumlah Responden (Ippd)\...,Berdasarkan pedoman pemantauan dan evaluasi SP...,Berdasarkan pedoman pemantauan dan evaluasi SP...
4,Dibawah ini merupakan domain-domain dari Arsit...,[Graph data:\n[INFO] Tidak ada hasil graph yan...,Domain arsitektur Manajemen SPBE tidak termasu...,Domain Arsitektur Manajemen SPBE tidak termasu...


In [24]:
# Save dataset
# df_ragas = pd.DataFrame(ragas_dataset, columns=["user_input", "retrieved_contexts", "response", "reference"])
df_ragas.to_csv("spbe5_graphrag_ragas.csv", index=False)

# Save as pickle
with open("spbe5_test_graphrag_ragas_dataset.pkl", "wb") as f:
    pickle.dump(ragas_dataset, f)

In [ ]:
ragas_dataset=pd.read_csv("spbe4_graphrag_ragas.csv")

In [25]:

"""
Menjalankan metrik RAGAS pada dataset GraphRAG
"""
print("📊 Menjalankan metrik RAGAS...")

# Create evaluation dataset
eval_dataset = EvaluationDataset.from_list(ragas_dataset)

embeddings_ollama = OllamaEmbeddings(
model="nomic-embed-text:latest",
)

# Integration Azure Open AI in Langchain
llm_openai_azure = AzureChatOpenAI(
    openai_api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.environ.get("41_AZURE_OPENAI_DEPLOYMENT"),
    openai_api_version=os.environ.get("41_AZURE_OPENAI_API_VERSION"),
)

# Initialize evaluators
evaluator_embedding = LangchainEmbeddingsWrapper(embeddings_ollama)
evaluator_llm = LangchainLLMWrapper(llm_openai_azure)

# Evaluasi dengan RAGAS
results = evaluate(
    dataset=eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embedding,
)

print("📊 Hasil Evaluasi RAGAS GraphRAG:")
results_df = results.to_pandas()
results_df.to_csv("evaluation2_spbe_graphrag.csv", index=False)
print(results_df)

📊 Menjalankan metrik RAGAS...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

📊 Hasil Evaluasi RAGAS GraphRAG:
                                          user_input  \
0  Dalam penerapan Manajemen Risiko SPBE, masing-...   
1  Jika "Rencana dan Anggaran SPBE Instansi Pusat...   
2  Berapa nilai indeks domain yang masih dibawah ...   
3  Sebuah IPPD melampirkan Peta Rencana SPBE yang...   
4  Dibawah ini merupakan domain-domain dari Arsit...   

                                  retrieved_contexts  \
0  [Graph data:\n🔹 Node: Jumlah Responden (Ippd)\...   
1  [Graph data:\n🔹 Node: Pelayanan Administrasi P...   
2  [Graph data:\n🔹 Node: Indeks Domain Manajemen\...   
3  [Graph data:\n🔹 Node: Jumlah Responden (Ippd)\...   
4  [Graph data:\n[INFO] Tidak ada hasil graph yan...   

                                            response  \
0  Permenpanrb No 5 Tahun 2020 Tentang Pedoman Ma...   
1  Berdasarkan pedoman pemantauan dan evaluasi SP...   
2  Nilai indeks domain yang masih dibawah target ...   
3  Berdasarkan pedoman pemantauan dan evaluasi SP...   
4  Domain ars

In [26]:
results_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision
0,"Dalam penerapan Manajemen Risiko SPBE, masing-...",[Graph data:\n🔹 Node: Jumlah Responden (Ippd)\...,Permenpanrb No 5 Tahun 2020 Tentang Pedoman Ma...,PermenPANRB No. 5 Tahun 2020 memberikan pedoma...,0.500000,0.604235,1.000000,1.0
1,"Jika ""Rencana dan Anggaran SPBE Instansi Pusat...",[Graph data:\n🔹 Node: Pelayanan Administrasi P...,Berdasarkan pedoman pemantauan dan evaluasi SP...,Berdasarkan pedoman pemantauan dan evaluasi SP...,0.625000,0.674860,0.285714,1.0
2,Berapa nilai indeks domain yang masih dibawah ...,[Graph data:\n🔹 Node: Indeks Domain Manajemen\...,Nilai indeks domain yang masih dibawah target ...,Nilai indeks domain yang masih di bawah target...,0.666667,0.865775,1.000000,1.0
3,Sebuah IPPD melampirkan Peta Rencana SPBE yang...,[Graph data:\n🔹 Node: Jumlah Responden (Ippd)\...,Berdasarkan pedoman pemantauan dan evaluasi SP...,Berdasarkan pedoman pemantauan dan evaluasi SP...,0.166667,0.742882,0.500000,1.0
4,Dibawah ini merupakan domain-domain dari Arsit...,[Graph data:\n[INFO] Tidak ada hasil graph yan...,Domain arsitektur Manajemen SPBE tidak termasu...,Domain Arsitektur Manajemen SPBE tidak termasu...,0.666667,0.950082,0.500000,1.0
